<a href="https://colab.research.google.com/github/twillixa/HEC/blob/main/Test/PDF_modifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
add_text_to_pdf.py
------------------
A beginner-friendly Python script that:
  1. Opens an existing PDF
  2. Adds custom text at a specific position on a chosen page
  3. Saves the result as a new PDF

Libraries used:
  - pypdf      : reads the original PDF pages
  - reportlab  : creates a small "overlay" PDF that contains only our new text
  Both are installable with:  pip install pypdf reportlab
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import io                                      # lets us work with files in memory
from pypdf import PdfReader, PdfWriter         # read & write PDF pages
from reportlab.pdfgen import canvas            # draw text / shapes on a PDF
from reportlab.lib.pagesizes import A4         # standard page size (width, height in points)


# ── Helper function ────────────────────────────────────────────────────────────
def create_text_overlay(text, x, y, page_size=A4, font="Helvetica", font_size=12, color=(1, 0, 0)):
    """
    Creates a tiny in-memory PDF that contains ONLY the text we want to add.
    We will later merge (overlay) this on top of the original PDF page.

    Parameters
    ----------
    text      : str   – the string you want to add to the PDF
    x         : float – horizontal position in points  (0 = left edge)
    y         : float – vertical position in points    (0 = bottom edge)
    page_size : tuple – (width, height) of the page, default A4
    font      : str   – font name (stick to built-in ones: Helvetica, Times-Roman, Courier)
    font_size : int   – size of the text in points
    color     : tuple – (R, G, B) each between 0 and 1  →  (1,0,0) = red

    Returns
    -------
    A PdfReader object containing the single-page overlay
    """

    # io.BytesIO() is like a file, but lives in RAM instead of on disk.
    # This avoids creating a temporary file.
    packet = io.BytesIO()

    # Create a canvas (think of it as a blank sheet of paper)
    c = canvas.Canvas(packet, pagesize=page_size)

    # Set the text color  (R, G, B  values between 0 and 1)
    c.setFillColorRGB(*color)

    # Choose the font and its size
    c.setFont(font, font_size)

    # Place the text at the (x, y) coordinates
    # IMPORTANT: in PDFs, y=0 is the BOTTOM of the page, y increases upward
    c.drawString(x, y, text)

    # "Save" the canvas — this finalises the page inside the BytesIO buffer
    c.save()

    # Rewind the buffer to the beginning so PdfReader can read it
    packet.seek(0)

    # Return a PdfReader built from our in-memory PDF
    return PdfReader(packet)


# ── Main function ──────────────────────────────────────────────────────────────
def add_text_to_pdf(
    input_path,
    output_path,
    text,
    page_number=0,        # which page to annotate  (0 = first page)
    x=100,                # horizontal position (points from left)
    y=100,                # vertical position   (points from bottom)
    font_size=12,
    color=(1, 0, 0),      # red by default so it stands out
):
    """
    Opens a PDF, adds text to a specified page, and saves the result.

    Parameters
    ----------
    input_path  : str  – path to the original PDF file
    output_path : str  – path where the modified PDF will be saved
    text        : str  – the text you want to insert
    page_number : int  – 0-based page index  (0 = first page, 1 = second, …)
    x, y        : float – position on the page in points
    font_size   : int  – text size
    color       : tuple – (R, G, B) colour, values 0–1
    """

    # Step 1 – Read the original PDF
    reader = PdfReader(input_path)
    total_pages = len(reader.pages)
    print(f" Opened '{input_path}'  →  {total_pages} page(s) found")

    # Validate the page number the user asked for
    if page_number >= total_pages or page_number < 0:
        raise ValueError(
            f"page_number={page_number} is out of range. "
            f"The PDF has {total_pages} page(s), so valid values are 0 to {total_pages - 1}."
        )

    # Step 2 – Create a PdfWriter (it will build our output PDF)
    writer = PdfWriter()

    # Step 3 – Loop over every page in the original PDF
    for i, page in enumerate(reader.pages):

        if i == page_number:
            # This is the page we want to modify.

            # Detect the page dimensions so the overlay matches exactly
            page_width  = float(page.mediabox.width)
            page_height = float(page.mediabox.height)
            page_size   = (page_width, page_height)

            # Create the overlay (a transparent page with just our text)
            overlay_reader = create_text_overlay(
                text=text,
                x=x,
                y=y,
                page_size=page_size,
                font_size=font_size,
                color=color,
            )
            overlay_page = overlay_reader.pages[0]

            # Merge the overlay ON TOP of the original page
            # merge_page() draws the overlay's content over the existing content
            page.merge_page(overlay_page)

            print(f"✅ Text added to page {i + 1}  at position  x={x}, y={y}")

        # Add the (possibly modified) page to the writer
        writer.add_page(page)

    # Step 4 – Write the finished PDF to disk
    with open(output_path, "wb") as output_file:   # "wb" = write in binary mode
        writer.write(output_file)

    print(f" Saved modified PDF  →  '{output_path}'")


# ── Entry point ────────────────────────────────────────────────────────────────
# This block only runs when you execute the script directly (not when imported).
if __name__ == "__main__":

    add_text_to_pdf(
        input_path="sample_input.pdf",       # ← change this to your PDF file
        output_path="output.pdf",            # ← the new file that will be created
        text="★ APPROVED by HR — 14 April 2026",
        page_number=0,                       # first page  (0-indexed)
        x=80,                                # 80 points from the left
        y=60,                                # 60 points from the bottom
        font_size=13,
        color=(0.8, 0, 0),                   # dark red
    )

In [ ]:
"""
add_text_to_pdf_v2.py
---------------------
An upgraded version of the PDF text-insertion script.
This version lets you add MULTIPLE strings at MULTIPLE positions,
across one or several pages, in a single call.

Install dependencies:
    pip install pypdf reportlab
"""

import io
from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas


# ── Helper: build one overlay page ────────────────────────────────────────────
def create_overlay_for_page(annotations, page_width, page_height):
    """
    Creates an in-memory PDF page that contains ALL the text annotations
    destined for a single PDF page.

    Parameters
    ----------
    annotations  : list of dicts, each dict has:
                     "text"      – the string to write
                     "x"         – horizontal position (points from left)
                     "y"         – vertical position   (points from bottom)
                     "font"      – font name  (default "Helvetica")
                     "font_size" – size in points (default 12)
                     "color"     – (R, G, B) tuple, values 0–1 (default black)
    page_width   : float – width of the target page
    page_height  : float – height of the target page

    Returns
    -------
    A pypdf Page object ready to be merged onto the original page.
    """
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=(page_width, page_height))

    for ann in annotations:
        # Pull values from the dict, with sensible defaults
        text      = ann["text"]
        x         = ann["x"]
        y         = ann["y"]
        font      = ann.get("font", "Helvetica")
        font_size = ann.get("font_size", 12)
        color     = ann.get("color", (0, 0, 0))   # default: black

        c.setFillColorRGB(*color)
        c.setFont(font, font_size)
        c.drawString(x, y, text)

    c.save()
    packet.seek(0)
    return PdfReader(packet).pages[0]


# ── Main function ──────────────────────────────────────────────────────────────
def add_texts_to_pdf(input_path, output_path, annotations):
    """
    Adds multiple text strings to specific positions across pages of a PDF.

    Parameters
    ----------
    input_path  : str  – path to the original PDF
    output_path : str  – path where the annotated PDF will be saved
    annotations : list of dicts, each dict has:
                    "text"      (required) – string to insert
                    "page"      (required) – 0-based page index
                    "x"         (required) – points from left edge
                    "y"         (required) – points from bottom edge
                    "font"      (optional) – default "Helvetica"
                    "font_size" (optional) – default 12
                    "color"     (optional) – (R,G,B) default black (0,0,0)

    Example annotation list
    -----------------------
    annotations = [
        {"text": "John Doe",          "page": 0, "x": 200, "y": 650},
        {"text": "12 Rue de Lausanne","page": 0, "x": 200, "y": 630},
        {"text": "John Doe",          "page": 1, "x": 100, "y": 700},
    ]
    """

    reader = PdfReader(input_path)
    total_pages = len(reader.pages)
    print(f"📄 Opened '{input_path}'  →  {total_pages} page(s)")

    # ── Group annotations by page number ──────────────────────────────────────
    # Instead of looping over annotations for every page (slow),
    # we build a dictionary:  { page_index: [list of annotations] }
    # so we can look up what to draw on each page in O(1).

    from collections import defaultdict
    annotations_by_page = defaultdict(list)

    for ann in annotations:
        p = ann["page"]
        if p < 0 or p >= total_pages:
            raise ValueError(
                f"Annotation targets page {p}, but the PDF only has "
                f"{total_pages} page(s) (valid: 0 – {total_pages - 1})."
            )
        annotations_by_page[p].append(ann)

    # ── Process each page ─────────────────────────────────────────────────────
    writer = PdfWriter()

    for i, page in enumerate(reader.pages):
        if i in annotations_by_page:
            page_width  = float(page.mediabox.width)
            page_height = float(page.mediabox.height)

            overlay_page = create_overlay_for_page(
                annotations_by_page[i], page_width, page_height
            )
            page.merge_page(overlay_page)

            n = len(annotations_by_page[i])
            print(f"  ✅ Page {i + 1}: added {n} annotation(s)")

        writer.add_page(page)

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(output_path, "wb") as f:
        writer.write(f)

    print(f"💾 Saved  →  '{output_path}'")


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Define your client data in one place ──────────────────────────────────
    # Changing it here automatically updates every annotation below.
    client = {
        "name":    "Sophie Martin",
        "address": "12 Rue de Lausanne, 1003 Lausanne",
        "date":    "14 April 2026",
    }

    # ── List every text you want to insert ────────────────────────────────────
    # Each dict = one piece of text.
    # "page" is 0-based  (0 = first page, 1 = second page, …)
    # x=0 is the LEFT edge,  y=0 is the BOTTOM edge.
    annotations = [
        # ── Page 1: header area ───────────────────────────────────────────────
        {
            "text":      client["name"],
            "page":      0,
            "x":         200,
            "y":         680,
            "font":      "Helvetica-Bold",
            "font_size": 13,
            "color":     (0, 0, 0.8),      # dark blue
        },
        {
            "text":      client["address"],
            "page":      0,
            "x":         200,
            "y":         660,
            "font":      "Helvetica",
            "font_size": 11,
            "color":     (0, 0, 0),        # black
        },
        # ── Page 1: signature block at the bottom ─────────────────────────────
        {
            "text":      f"Employee: {client['name']}",
            "page":      0,
            "x":         80,
            "y":         130,
            "font_size": 11,
        },
        {
            "text":      f"Date: {client['date']}",
            "page":      0,
            "x":         80,
            "y":         110,
            "font_size": 11,
        },
    ]

    add_texts_to_pdf(
        input_path="sample_input.pdf",
        output_path="output_v2.pdf",
        annotations=annotations,
    )